# POWSM Fine-tuning — Method Comparison & Visualization

Single notebook that:
1. Trains 3 PEFT methods (LoRA r=8, LoRA r=32, DoRA r=8)
2. Evaluates baseline + all methods on the test split
3. Generates loss curves, PER bar chart, per-phoneme heatmap, and summary markdown

**Resume support:** methods with an existing `history.json` are skipped — Colab disconnects don't lose work.

Estimated total runtime on T4: ~6–9 hours for all 3 training runs + eval + visualization.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !apt-get install -q -y cmake sox libsox-dev libsndfile1-dev
    !pip install -q --upgrade pip setuptools wheel
    !pip install -q espnet espnet-model-zoo "peft==0.13.2" soundfile librosa omegaconf tqdm matplotlib seaborn pandas

In [ ]:
# ============================================================
# ⚙️  CONFIGURATION
# ============================================================

DRIVE_ROOT = "/content/drive/MyDrive/senior"
CHUNKS_DIR = "/content/turkish_chunks"   # local SSD; or DRIVE_ROOT + '/turkish_chunks'
RUNS_DIR    = "/content/drive/MyDrive/senior/runs"
RESULTS_DIR = "/content/drive/MyDrive/senior/results"

MODEL_ID  = "espnet/powsm"
LANG_SYM  = "<unk>"
TASK_SYM  = "<pr>"

# Training hyperparameters (shared across methods for fair comparison)
EPOCHS     = 15
BATCH_SIZE = 2     # T4 = 2; A100 can use 8
LR         = 1e-4
GRAD_CLIP  = 1.0
SEED       = 42

# Methods to train (delete entries to skip; resume support is automatic)
METHODS = {
    "lora_r8":  dict(r=8,  lora_alpha=16, use_dora=False),
    "lora_r32": dict(r=32, lora_alpha=64, use_dora=False),
    "dora_r8":  dict(r=8,  lora_alpha=16, use_dora=True),
}
COMMON_LORA_KW = dict(
    target_modules=["linear_q", "linear_k", "linear_v", "linear_out"],
    lora_dropout=0.1,
    bias="none",
)

# Turkish phones to highlight in heatmap
TUR_PHONES = ["ɯ", "ø", "œ", "y", "ç", "ɟ", "ɣ", "ɰ"]
# ============================================================

In [ ]:
# Drive mount + sys.path setup (runs before any imports that need local modules)
import sys
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # turkish_lora_util.py lives in DRIVE_ROOT
    sys.path.insert(0, DRIVE_ROOT)
    # assessment package lives in DRIVE_ROOT/mod/
    sys.path.insert(0, str(Path(DRIVE_ROOT) / "mod"))
else:
    # Local: auto-detect repo layout
    _here = Path.cwd().resolve()
    FT = _here if (_here / "turkish_lora_util.py").is_file() else _here.parent
    REPO_ROOT = FT.parents[1]          # sig/fine-tune → sig → senior (repo root)
    sys.path.insert(0, str(FT))        # turkish_lora_util.py
    sys.path.insert(0, str(REPO_ROOT / "mod"))  # assessment package
    # Override string paths from config cell
    CHUNKS_DIR   = str(FT / "data" / "turkish_chunks")
    RUNS_DIR     = str(FT / "lora_runs")
    RESULTS_DIR  = str(FT / "results")

# Convert to Path objects (used throughout the notebook)
CHUNKS_DIR   = Path(CHUNKS_DIR)
RUNS_DIR     = Path(RUNS_DIR)
RESULTS_DIR  = Path(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("CHUNKS_DIR :", CHUNKS_DIR,  "| exists:", CHUNKS_DIR.is_dir())
print("RUNS_DIR   :", RUNS_DIR,    "| exists:", RUNS_DIR.is_dir())
print("RESULTS_DIR:", RESULTS_DIR, "| exists:", RESULTS_DIR.is_dir())

In [ ]:
# Extract chunks to local SSD on Colab — much faster I/O than reading from Drive
# Handles the case where Drive holds a tar-disguised-as-zip (use 7z, which auto-detects).
if IN_COLAB and not CHUNKS_DIR.is_dir():
    archive = Path(DRIVE_ROOT) / "turkish_chunks.zip"
    if archive.is_file():
        print(f"Extracting {archive} → /content/ (using 7z) …")
        !apt-get install -q -y p7zip-full
        !7z x "{archive}" -o/content/ -y > /dev/null
    else:
        # Fallback: copy the folder directly from Drive
        drive_folder = Path(DRIVE_ROOT) / "turkish_chunks"
        if drive_folder.is_dir():
            print(f"Copying {drive_folder} → /content/ …")
            import subprocess
            subprocess.run(["cp", "-r", str(drive_folder), "/content/"], check=True)
        else:
            raise FileNotFoundError(
                f"Neither {archive} nor {drive_folder} exists. "
                "Upload turkish_chunks (folder or .zip) to MyDrive/senior/."
            )
    print(f"Done. {len(list(CHUNKS_DIR.glob('*.wav')))} WAV files on local SSD.")
elif IN_COLAB:
    print(f"Already extracted: {len(list(CHUNKS_DIR.glob('*.wav')))} WAV files.")

In [ ]:
# Extract chunks to local SSD on Colab — much faster I/O than reading from Drive
# Handles the case where Drive holds a tar-disguised-as-zip (use 7z, which auto-detects).
if IN_COLAB and not CHUNKS_DIR.is_dir():
    archive = Path(DRIVE_ROOT) / "turkish_chunks.zip"
    if archive.is_file():
        print(f"Extracting {archive} → /content/ (using 7z) …")
        !apt-get install -q -y p7zip-full
        !7z x "{archive}" -o/content/ -y > /dev/null
    else:
        # Fallback: copy the folder directly from Drive
        drive_folder = Path(DRIVE_ROOT) / "turkish_chunks"
        if drive_folder.is_dir():
            print(f"Copying {drive_folder} → /content/ …")
            import subprocess
            subprocess.run(["cp", "-r", str(drive_folder), "/content/"], check=True)
        else:
            raise FileNotFoundError(
                f"Neither {archive} nor {drive_folder} exists. "
                "Upload turkish_chunks (folder or .zip) to MyDrive/senior/."
            )
    print(f"Done. {len(list(CHUNKS_DIR.glob('*.wav')))} WAV files on local SSD.")
elif IN_COLAB:
    print(f"Already extracted: {len(list(CHUNKS_DIR.glob('*.wav')))} WAV files.")

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory // 1024**3
    print(f"GPU: {name} | {vram} GB VRAM")
else:
    print("WARNING: no GPU — training will be very slow")
torch.manual_seed(SEED)

In [ ]:
import copy
import json
import time
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from omegaconf import OmegaConf
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from espnet2.bin.s2t_inference import Speech2Text
from espnet2.train.preprocessor import S2TPreprocessor
from espnet2.train.collate_fn import common_collate_fn
from espnet2.torch_utils.device_funcs import to_device
from peft import LoraConfig, get_peft_model

from turkish_lora_util import patch_speech2text_lora
from assessment.edit_distance import edit_operations

print("imports ok")

## Data pipeline

In [ ]:
# Load POWSM once to extract preprocessor config; reused for all training runs
s2t_init = Speech2Text.from_pretrained(
    MODEL_ID, device=DEVICE, lang_sym=LANG_SYM, task_sym=TASK_SYM
)
args = s2t_init.s2t_train_args
_raw = args.preprocessor_conf
pc = copy.deepcopy(_raw) if isinstance(_raw, dict) else OmegaConf.to_container(_raw, resolve=True)
for k in ("token_list", "token_type", "bpemodel", "text_cleaner", "g2p_type", "non_linguistic_symbols"):
    pc.pop(k, None)
pc["speech_length"] = 20.0

_prep_kw = dict(
    token_type=args.token_type,
    token_list=list(s2t_init.s2t_model.token_list),
    bpemodel=args.bpemodel,
    text_cleaner=args.cleaner,
    g2p_type=getattr(args, "g2p", None),
    non_linguistic_symbols=getattr(args, "non_linguistic_symbols", None),
    rir_scp=getattr(args, "rir_scp", None),
    rir_apply_prob=getattr(args, "rir_apply_prob", 1.0),
    noise_scp=getattr(args, "noise_scp", None),
    noise_apply_prob=getattr(args, "noise_apply_prob", 1.0),
    noise_db_range=getattr(args, "noise_db_range", "13_15"),
    short_noise_thres=getattr(args, "short_noise_thres", 0.5),
    speech_volume_normalize=getattr(args, "speech_volume_normalize", None),
)

prep_train = S2TPreprocessor(train=True,  **_prep_kw, **pc)
prep_eval  = S2TPreprocessor(train=False, **_prep_kw, **pc)
for p in (prep_train, prep_eval):
    p.text_prev_apply_prob = 1.0
    p.time_apply_prob = 1.0

# Free the init model — each method will load fresh
del s2t_init; torch.cuda.empty_cache()
print("preprocessors ready")

In [ ]:
def build_pr_text(phones):
    return f"{LANG_SYM}{TASK_SYM}<notimestamps> " + "".join(f"/{p}/" for p in phones)


class TurkishChunkS2T(Dataset):
    def __init__(self, manifest_path, data_dir, prep):
        self.items = json.loads(Path(manifest_path).read_text(encoding="utf-8"))
        self.data_dir = Path(data_dir)
        self.prep = prep

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        speech, sr = sf.read(self.data_dir / f"{item['id']}.wav")
        if sr != 16000: raise ValueError(sr)
        text = build_pr_text(item["phones"])
        return item["id"], self.prep(item["id"], {
            "speech": np.asarray(speech, dtype=np.float32),
            "text": text, "text_prev": "<na>", "text_ctc": text,
        })


def collate_s2t(samples):
    return common_collate_fn(samples, int_pad_value=-1)


train_ds = TurkishChunkS2T(CHUNKS_DIR / "train.json", CHUNKS_DIR, prep_train)
val_ds   = TurkishChunkS2T(CHUNKS_DIR / "val.json",   CHUNKS_DIR, prep_eval)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_s2t, num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_s2t, num_workers=0)
test_items = json.loads((CHUNKS_DIR / "test.json").read_text(encoding="utf-8"))
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_items)}")

## Training all methods

Each method gets a fresh base model (no adapter contamination). Skips methods whose `history.json` already exists.

In [ ]:
def train_one_method(method_name, lora_kwargs):
    out_dir = RUNS_DIR / method_name
    history_path = out_dir / "history.json"
    if history_path.is_file():
        print(f"[skip] {method_name} already done")
        return json.loads(history_path.read_text())

    out_dir.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(SEED)

    s2t = Speech2Text.from_pretrained(MODEL_ID, device=DEVICE, lang_sym=LANG_SYM, task_sym=TASK_SYM)
    base = s2t.s2t_model
    base.train()
    cfg = LoraConfig(**lora_kwargs, **COMMON_LORA_KW)
    model = get_peft_model(base, cfg)

    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"  trainable: {n_train:,} / {n_total:,} ({100*n_train/n_total:.2f}%)")

    optimizer = AdamW((p for p in model.parameters() if p.requires_grad), lr=LR, weight_decay=0.01)
    scheduler = OneCycleLR(optimizer, max_lr=LR, steps_per_epoch=max(1, len(train_dl)), epochs=EPOCHS)

    history = {"method": method_name, "trainable_params": n_train, "total_params": n_total, "epochs": []}
    best_val = float("inf")

    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train()
        train_acc = defaultdict(list)
        for _ids, batch in tqdm(train_dl, desc=f"{method_name} ep{epoch+1}", leave=False):
            batch = to_device(batch, DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss, stats, _ = model(**batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step(); scheduler.step()
            for k, v in stats.items():
                if v is not None:
                    train_acc[k].append(float(v))

        model.eval()
        val_acc = defaultdict(list)
        with torch.no_grad():
            for _ids, batch in val_dl:
                batch = to_device(batch, DEVICE)
                _, stats, _ = model(**batch)
                for k, v in stats.items():
                    if v is not None:
                        val_acc[k].append(float(v))

        rec = {"epoch": epoch + 1, "wall_seconds": time.time() - t0}
        for k, vs in train_acc.items(): rec[f"train_{k}"] = sum(vs) / len(vs)
        for k, vs in val_acc.items():   rec[f"val_{k}"]   = sum(vs) / len(vs)
        history["epochs"].append(rec)
        print(f"  ep{epoch+1}: train_loss={rec.get('train_loss', float('nan')):.3f} "
              f"val_loss={rec.get('val_loss', float('nan')):.3f} "
              f"val_acc={rec.get('val_acc', float('nan')):.3f}")

        if rec.get("val_loss", float("inf")) < best_val:
            best_val = rec["val_loss"]
            (out_dir / "best").mkdir(parents=True, exist_ok=True)
            model.save_pretrained(out_dir / "best")

    history["best_val_loss"] = best_val
    history_path.write_text(json.dumps(history, indent=2))

    del model, base, s2t
    torch.cuda.empty_cache()
    return history

In [ ]:
all_histories = {}
for name, kw in METHODS.items():
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    all_histories[name] = train_one_method(name, kw)
print("\nAll training done.")

## Evaluation: baseline + all trained methods

In [ ]:
def parse_pr_tokens(raw):
    if "<notimestamps>" in raw:
        raw = raw.split("<notimestamps>", 1)[1]
    tokens = []
    for part in raw.strip().split("//"):
        part = part.strip().strip("/")
        if part:
            tokens.append(part)
    return tokens


def eval_method(method_name=None):
    """method_name=None evaluates baseline POWSM (no adapter)."""
    s2t = Speech2Text.from_pretrained(MODEL_ID, device=DEVICE, lang_sym=LANG_SYM, task_sym=TASK_SYM)
    if method_name is not None:
        patch_speech2text_lora(s2t, RUNS_DIR / method_name / "best")

    pers = []
    confusion = defaultdict(lambda: defaultdict(int))
    by_speaker = defaultdict(list)
    by_task = defaultdict(list)
    ref_phone_counts = Counter()

    for item in tqdm(test_items, desc=f"eval {method_name or 'baseline'}"):
        ref = item["phones"]
        for p in ref: ref_phone_counts[p] += 1
        speech, _ = sf.read(CHUNKS_DIR / f"{item['id']}.wav")
        raw = s2t(np.asarray(speech, dtype=np.float32), text_prev="<na>")[0][0]
        pred = parse_pr_tokens(raw)
        ops = edit_operations(pred, ref)
        per = len(ops) / max(len(ref), 1)
        pers.append(per)
        by_speaker[item["speaker"]].append(per)
        by_task[item["task"]].append(per)
        # edit_operations returns: ("substitute", pos, target, actual), ("insert", pos, actual), ("delete", pos, target)
        for op in ops:
            if op[0] == "substitute":
                # confusion[ref_phone][pred_phone] += 1
                confusion[op[2]][op[3]] += 1
            elif op[0] == "delete":
                # missing from prediction (ref had it, pred didn't)
                confusion[op[2]]["<del>"] += 1
            elif op[0] == "insert":
                # extra in prediction (pred had it, ref didn't)
                confusion["<ins>"][op[2]] += 1

    del s2t; torch.cuda.empty_cache()

    return {
        "method": method_name or "baseline",
        "mean_per": float(np.mean(pers)),
        "std_per":  float(np.std(pers)),
        "sem_per":  float(np.std(pers) / np.sqrt(len(pers))),
        "per_by_task":    {k: float(np.mean(v)) for k, v in by_task.items()},
        "per_by_speaker": {k: float(np.mean(v)) for k, v in by_speaker.items()},
        "confusion": {k: dict(v) for k, v in confusion.items()},
        "ref_phone_counts": dict(ref_phone_counts),
    }

In [ ]:
all_eval = {"baseline": eval_method(None)}
for name in METHODS:
    all_eval[name] = eval_method(name)

(RESULTS_DIR / "all_eval.json").write_text(
    json.dumps(all_eval, indent=2, ensure_ascii=False), encoding="utf-8"
)
for name, e in all_eval.items():
    print(f"{name:<12} mean PER = {e['mean_per']:.4f}  ({e['mean_per']*100:.1f}%)")

## Visualization

In [ ]:
# Loss curves — 2x2 grid: total loss, loss_att, loss_ctc, val_acc
sns.set_style("whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
panels = [
    (axes[0,0], "loss",     "Total loss (CTC+attn)"),
    (axes[0,1], "loss_att", "Attention loss"),
    (axes[1,0], "loss_ctc", "CTC loss"),
    (axes[1,1], "acc",      "Decoder accuracy"),
]
colors = sns.color_palette("tab10", n_colors=len(METHODS))

for ax, key, title in panels:
    for (name, hist), c in zip(all_histories.items(), colors):
        epochs = [e["epoch"] for e in hist["epochs"]]
        train_vals = [e.get(f"train_{key}") for e in hist["epochs"]]
        val_vals   = [e.get(f"val_{key}")   for e in hist["epochs"]]
        if any(v is not None for v in train_vals):
            ax.plot(epochs, train_vals, color=c, linestyle="-",  label=f"{name} train")
        if any(v is not None for v in val_vals):
            ax.plot(epochs, val_vals,   color=c, linestyle="--", label=f"{name} val")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(key)
    ax.legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved:", RESULTS_DIR / "loss_curves.png")

In [ ]:
# PER bar chart + markdown table
method_order = ["baseline"] + list(METHODS.keys())
rows = []
for m in method_order:
    e = all_eval[m]
    hist = all_histories.get(m)
    train_h = (sum(ep["wall_seconds"] for ep in hist["epochs"]) / 3600.0) if hist else 0.0
    n_train = hist["trainable_params"] if hist else 0
    rows.append({
        "method": m,
        "trainable_params": n_train,
        "mean_per": e["mean_per"],
        "sem_per":  e["sem_per"],
        "per_task1": e["per_by_task"].get("task1", float("nan")),
        "per_task2": e["per_by_task"].get("task2", float("nan")),
        "train_hours": train_h,
    })
df = pd.DataFrame(rows)
baseline_per = df.loc[df["method"] == "baseline", "mean_per"].iloc[0]
df["delta_pp"] = (df["mean_per"] - baseline_per) * 100

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(df["method"], df["mean_per"] * 100, yerr=df["sem_per"] * 100,
              capsize=4, color=["gray"] + list(sns.color_palette("tab10", len(METHODS))))
for b, v in zip(bars, df["mean_per"] * 100):
    ax.annotate(f"{v:.1f}%", xy=(b.get_x() + b.get_width()/2, v),
                xytext=(0, 4), textcoords="offset points", ha="center", fontsize=10)
ax.set_ylabel("Phone Error Rate (%)")
ax.set_title("PER on test split (lower is better)")
ax.set_ylim(0, max(df["mean_per"] * 100) * 1.2)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "per_bar.png", dpi=150, bbox_inches="tight")
plt.show()

# Markdown table
md_lines = [
    "| Method   | Trainable | Mean PER | Δ vs base | Read-aloud | Interview | Train (h) |",
    "|----------|-----------|----------|-----------|------------|-----------|-----------|",
]
for _, r in df.iterrows():
    delta = "—" if r["method"] == "baseline" else f"{r['delta_pp']:+.2f} pp"
    md_lines.append(
        f"| {r['method']:<8} | {r['trainable_params']:>9,} | {r['mean_per']*100:>7.2f}% | "
        f"{delta:>9} | {r['per_task1']*100:>9.2f}% | {r['per_task2']*100:>8.2f}% | {r['train_hours']:>8.2f} |"
    )
md_table = "\n".join(md_lines)
(RESULTS_DIR / "per_table.md").write_text(md_table, encoding="utf-8")
df.to_csv(RESULTS_DIR / "per_table.csv", index=False)
print(md_table)

In [ ]:
# Per-phoneme error-rate heatmap
ref_counts = all_eval["baseline"]["ref_phone_counts"]
top_freq = [p for p, _ in Counter(ref_counts).most_common(12)]
phones_to_show = list(dict.fromkeys(TUR_PHONES + top_freq))   # preserves order, dedups
phones_to_show = [p for p in phones_to_show if p in ref_counts]

matrix = np.zeros((len(method_order), len(phones_to_show)))
annots = np.zeros_like(matrix, dtype=int)
for i, m in enumerate(method_order):
    conf = all_eval[m]["confusion"]
    for j, ph in enumerate(phones_to_show):
        errs = sum(conf.get(ph, {}).values())
        denom = ref_counts.get(ph, 0)
        matrix[i, j] = errs / denom if denom else 0
        annots[i, j] = errs

fig, ax = plt.subplots(figsize=(max(8, len(phones_to_show) * 0.6), 0.6 + 0.6*len(method_order)))
sns.heatmap(matrix, annot=annots, fmt="d", cmap="Reds",
            xticklabels=[f"/{p}/" for p in phones_to_show],
            yticklabels=method_order, ax=ax, cbar_kws={"label": "Error rate per phone"})
ax.set_title("Per-phoneme error rate (annotation = error count)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "phoneme_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Auto-generated summary.md
lines = [
    "# POWSM Fine-tuning Comparison Results",
    "",
    "## Setup",
    f"- Test set: **{len(test_items)} chunks** from held-out speakers",
    f"- Tasks covered: read-aloud + elicited interview",
    f"- Methods: baseline + {', '.join(METHODS.keys())}",
    f"- Training: {EPOCHS} epochs, lr={LR}, batch_size={BATCH_SIZE}, seed={SEED}",
    "",
    "## Headline numbers",
    "",
    md_table,
    "",
    "## Phones with biggest improvement",
]

# Find phones where best fine-tuned method beats baseline most
best_method = min((m for m in METHODS), key=lambda m: all_eval[m]["mean_per"])
improvements = []
for ph in phones_to_show:
    base_errs = sum(all_eval["baseline"]["confusion"].get(ph, {}).values())
    best_errs = sum(all_eval[best_method]["confusion"].get(ph, {}).values())
    denom = ref_counts.get(ph, 0)
    if denom:
        improvements.append((ph, base_errs - best_errs, base_errs, best_errs, denom))
improvements.sort(key=lambda t: -t[1])
for ph, delta, base_errs, best_errs, denom in improvements[:5]:
    if delta > 0:
        lines.append(
            f"- /{ph}/ ({denom} occurrences): baseline {base_errs} errors → "
            f"{best_method} {best_errs} errors  (Δ −{delta})"
        )

lines += ["", "## Per-method best epoch", ""]
for m, h in all_histories.items():
    if h.get("epochs"):
        best_ep = min(h["epochs"], key=lambda e: e.get("val_loss", float("inf")))
        lines.append(f"- **{m}**: best at epoch {best_ep['epoch']} "
                     f"(val_loss={best_ep.get('val_loss', float('nan')):.3f})")

lines += ["", "## Files", ""]
for fn in ("loss_curves.png", "per_bar.png", "phoneme_heatmap.png",
           "per_table.md", "per_table.csv", "all_eval.json"):
    lines.append(f"- `{fn}`")

summary_md = "\n".join(lines)
(RESULTS_DIR / "summary.md").write_text(summary_md, encoding="utf-8")
print(summary_md)